# Retail Credit Default Early Warning

## Interactive exploration

This Snowflake Workspace notebook investigates the  account portfolio data. The target is whether an existing account reaches the default state within 90 days.

The intended use is portfolio monitoring and case prioritisation.

## 1. Create Snowpark session and execution context

In [ ]:
import modin.pandas as pd
import snowflake.snowpark.modin.plugin

from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.use_role("CRISK_DEMO_DEVELOPER")
session.use_warehouse("CRISK_DEMO_WH")
session.use_database("CRISK_DEMO_DB")

Check context using SQL

In [ ]:
%%sql -r execution_context
SELECT
  CURRENT_ROLE() AS ACTIVE_ROLE,
  CURRENT_WAREHOUSE() AS ACTIVE_WAREHOUSE,
  CURRENT_DATABASE() AS ACTIVE_DATABASE,
  CURRENT_SCHEMA() AS ACTIVE_SCHEMA;

## 2. Inspect the data contract before choosing features

The training relation has one row per account and observation month with finalised ground truth. Start by examining its schema and a bounded sample rather than assuming which columns should enter a model.

In [ ]:
%%sql -r training_schema
DESCRIBE VIEW CRISK_DEMO_DB.RAW.TRAINING_BASE;

In [ ]:
traning_base_pd =  pd.read_snowflake("CRISK_DEMO_DB.RAW.TRAINING_BASE")

traning_base_pd.sort_values(["OBSERVATION_DATE", "ACCOUNT_ID"]).head(20)

The sample should show operational account attributes, behavioural measures, a finalised 90-day outcome, and its finality date. Account ID, observation date, and outcome finality date describe identity or timing; they are not automatically modelling features.

In [ ]:
training_row_count = len(traning_base_pd)
account_count = traning_base_pd["ACCOUNT_ID"].nunique()
first_observation = traning_base_pd["OBSERVATION_DATE"].min()
last_observation = traning_base_pd["OBSERVATION_DATE"].max()
observation_month_count = traning_base_pd["OBSERVATION_DATE"].nunique()
unique_key_count = len(traning_base_pd[["ACCOUNT_ID", "OBSERVATION_DATE"]].drop_duplicates())
duplicate_key_count = training_row_count - unique_key_count
non_final_label_count = (traning_base_pd["OUTCOME_FINALITY_DATE"] > pd.Timestamp("2026-09-01")).sum()

contract_summary = pd.DataFrame({
    "TRAINING_ROW_COUNT": [training_row_count],
    "ACCOUNT_COUNT": [account_count],
    "FIRST_OBSERVATION": [first_observation],
    "LAST_OBSERVATION": [last_observation],
    "OBSERVATION_MONTH_COUNT": [observation_month_count],
    "DUPLICATE_KEY_COUNT": [duplicate_key_count],
    "NON_FINAL_LABEL_COUNT": [non_final_label_count],
})

if duplicate_key_count != 0:
    raise ValueError(f"Expected unique account-date keys; found {duplicate_key_count} duplicates")
if non_final_label_count != 0:
    raise ValueError(f"Training data contains {non_final_label_count} non-final labels")

contract_summary

In [ ]:
null_counts = traning_base_pd.isnull().sum()
null_profile = null_counts.to_frame(name="NULL_COUNT").T
null_profile.columns = [f"{col}_NULLS" for col in null_profile.columns]

if null_counts.sum() != 0:
    raise ValueError(f"Training data contains {null_counts.sum()} null values")

null_profile

The contract checks now stop execution if account-date keys are duplicated, labels are not final, or required values are missing. With the modelling population established, the next question is whether the target is sufficiently supported and stable over the period available for development.

## 3. Understand target availability and time

Credit outcomes arrive after the 90-day window. The full outcome table therefore contains both finalised historical labels and pending recent observations. Training must use only the finalised relation.

In [ ]:
default_outcome_pd = pd.read_snowflake("CRISK_DEMO_DB.RAW.DEFAULT_OUTCOME")

outcome_stats = default_outcome_pd.groupby("OUTCOME_STATUS").agg(
    OUTCOME_COUNT=("OUTCOME_STATUS", "count"),
    FIRST_OBSERVATION=("OBSERVATION_DATE", "min"),
    LAST_OBSERVATION=("OBSERVATION_DATE", "max"),
).reset_index()

finalised_pre = default_outcome_pd[
    (default_outcome_pd["OUTCOME_STATUS"] == "FINALISED")
    & (default_outcome_pd["OBSERVATION_DATE"] < pd.Timestamp("2026-01-01"))
    & (default_outcome_pd["OUTCOME_FINALITY_DATE"] < pd.Timestamp("2026-01-01"))
]
pre_holdout_rate = pd.DataFrame({
    "OUTCOME_STATUS": ["FINALISED"],
    "PRE_HOLDOUT_DEFAULT_RATE": [finalised_pre["DEFAULT_WITHIN_90D"].mean()],
})
outcome_stats = outcome_stats.merge(pre_holdout_rate, on="OUTCOME_STATUS", how="left")
outcome_stats.sort_values("OUTCOME_STATUS")

In [ ]:
development_cutoff = pd.Timestamp("2025-07-01")
validation_cutoff = pd.Timestamp("2026-01-01")

eligible_development = (
    (traning_base_pd["OBSERVATION_DATE"] < development_cutoff)
    & (traning_base_pd["OUTCOME_FINALITY_DATE"] < development_cutoff)
)
eligible_validation = (
    (traning_base_pd["OBSERVATION_DATE"] >= development_cutoff)
    & (traning_base_pd["OBSERVATION_DATE"] < validation_cutoff)
    & (traning_base_pd["OUTCOME_FINALITY_DATE"] < validation_cutoff)
)
pre_holdout = traning_base_pd[eligible_development | eligible_validation]

embargo_summary = pd.DataFrame({
    "ELIGIBLE_PRE_HOLDOUT_COUNT": [len(pre_holdout)],
    "PURGED_AT_BOUNDARIES_COUNT": [
        len(traning_base_pd[traning_base_pd["OBSERVATION_DATE"] < validation_cutoff]) - len(pre_holdout)
    ],
})
embargo_summary

In [ ]:
monthly_target = pre_holdout.groupby("OBSERVATION_DATE").agg(
    TRAINING_ROW_COUNT=("OBSERVATION_DATE", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
    DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index().sort_values("OBSERVATION_DATE")
monthly_target

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

monthly_target_plot = monthly_target.copy()
monthly_target_plot["OBSERVATION_DATE"] = pd.to_datetime(monthly_target_plot["OBSERVATION_DATE"])

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
sns.lineplot(data=monthly_target_plot, x="OBSERVATION_DATE", y="DEFAULT_RATE", marker="o", ax=axes[0])
axes[0].set_title("Pre-holdout 90-day default rate by observation month")
axes[0].set_ylabel("Default rate")

sns.barplot(data=monthly_target_plot, x="OBSERVATION_DATE", y="TRAINING_ROW_COUNT", color="steelblue", ax=axes[1])
axes[1].set_title("Pre-holdout training observations by month")
axes[1].set_xlabel("Observation month")
axes[1].set_ylabel("Training rows")
axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

The pre-holdout monthly series confirms that random row splitting would place the same accounts and adjacent months on both sides of the split. The target is also imbalanced. These observations lead to two decisions: preserve time order with a label-finality embargo, and inspect feature signal using development and validation only. Outcomes from 2026 onward remain untouched until final Phase 3 evaluation.

## 4. Discover candidate features from the observed schema

Classify the available columns only after inspecting the schema. This makes exclusions and candidate roles visible rather than embedding a preselected list inside training code.

In [ ]:
dtypes_series = traning_base_pd.dtypes
col_info = []
for i, (col, dtype) in enumerate(dtypes_series.items()):
    dtype_str = str(dtype)
    if col == "ACCOUNT_ID":
        role = "IDENTIFIER_EXCLUDE"
    elif col in ("OBSERVATION_DATE", "OUTCOME_FINALITY_DATE"):
        role = "TIME_CONTROL_EXCLUDE"
    elif col == "DEFAULT_WITHIN_90D":
        role = "TARGET"
    elif "object" in dtype_str or "string" in dtype_str:
        role = "CATEGORICAL_CANDIDATE"
    else:
        role = "NUMERIC_CANDIDATE"
    col_info.append({"COLUMN_NAME": col, "DATA_TYPE": dtype_str, "ORDINAL_POSITION": i + 1, "PROVISIONAL_ROLE": role})

column_roles = pd.DataFrame(col_info)
column_roles

The schema-driven catalogue separates identifiers, timing controls, the target, and candidate predictors. Before examining target relationships, inspect ranges and tails to distinguish plausible operational values from errors and to identify transformations that may be needed later.

## 5. Inspect ranges, tails, and redundancy

Start with univariate ranges and quantiles. This determines whether the candidate fields behave like valid account measures before asking whether they separate the target. The same section then checks numeric redundancy so the first model is not built from several near-duplicate signals.

In [ ]:
numeric_cols = [
    "ACCOUNT_AGE_MONTHS", "CREDIT_LIMIT", "UTILISATION_RATIO",
    "MONTHLY_INCOME_ESTIMATE", "PAYMENT_AMOUNT_30D", "MISSED_PAYMENT_30D",
    "MISSED_PAYMENTS_6M", "IN_ARREARS_30D", "CUSTOMER_CONTACTS_90D",
]

numeric_profile = pre_holdout[numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
numeric_profile.index.name = "FEATURE"
numeric_profile = numeric_profile.reset_index().sort_values("FEATURE")
numeric_profile

In [ ]:
tail_checks = pd.DataFrame({
    "CHECK": [
        "UTILISATION_ABOVE_100_PERCENT",
        "UTILISATION_AT_OR_BELOW_ZERO",
        "NON_POSITIVE_INCOME",
        "NON_POSITIVE_PAYMENT",
        "MISSED_PAYMENTS_OUTSIDE_0_TO_3",
    ],
    "OBSERVATION_COUNT": [
        (pre_holdout["UTILISATION_RATIO"] > 1.0).sum(),
        (pre_holdout["UTILISATION_RATIO"] <= 0.0).sum(),
        (pre_holdout["MONTHLY_INCOME_ESTIMATE"] <= 0).sum(),
        (pre_holdout["PAYMENT_AMOUNT_30D"] <= 0).sum(),
        ((pre_holdout["MISSED_PAYMENTS_6M"] < 0) | (pre_holdout["MISSED_PAYMENTS_6M"] > 3)).sum(),
    ],
})
tail_checks

In [ ]:
numeric_correlation = pre_holdout[numeric_cols].corr(method="pearson")
numeric_correlation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(numeric_correlation, cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Pre-holdout numeric feature correlation")
plt.tight_layout()
plt.show()

In [ ]:
target_profile = pre_holdout.groupby("DEFAULT_WITHIN_90D").agg(
    OBSERVATION_COUNT=("DEFAULT_WITHIN_90D", "count"),
    AVG_UTILISATION_RATIO=("UTILISATION_RATIO", "mean"),
    AVG_MONTHLY_INCOME=("MONTHLY_INCOME_ESTIMATE", "mean"),
    AVG_PAYMENT_AMOUNT_30D=("PAYMENT_AMOUNT_30D", "mean"),
    MISSED_PAYMENT_RATE_30D=("MISSED_PAYMENT_30D", "mean"),
    AVG_MISSED_PAYMENTS_6M=("MISSED_PAYMENTS_6M", "mean"),
    ARREARS_RATE_30D=("IN_ARREARS_30D", "mean"),
    AVG_CUSTOMER_CONTACTS_90D=("CUSTOMER_CONTACTS_90D", "mean"),
).reset_index().sort_values("DEFAULT_WITHIN_90D")
target_profile

In [ ]:
import numpy as np

util_data = pre_holdout.copy()
util_data["UTILISATION_BIN"] = (util_data["UTILISATION_RATIO"] * 10 // 1) / 10
utilisation_distribution = util_data.groupby(["UTILISATION_BIN", "DEFAULT_WITHIN_90D"]).agg(
    OBSERVATION_COUNT=("DEFAULT_WITHIN_90D", "count"),
).reset_index().sort_values(["UTILISATION_BIN", "DEFAULT_WITHIN_90D"])
utilisation_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=utilisation_distribution,
    x="UTILISATION_BIN",
    y="OBSERVATION_COUNT",
    hue="DEFAULT_WITHIN_90D",
    ax=ax,
)
ax.set_title("Utilisation distribution by finalised outcome")
ax.set_xlabel("Utilisation ratio bin")
ax.set_ylabel("Account-month observations")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

Differences between outcome classes indicate useful signal but do not establish causality. The logarithmic count axis keeps the minority default class visible. Scaling and transformation decisions remain deferred until the modelling pipeline is chosen.

The univariate profiles show whether values are operationally plausible, while the correlation matrix identifies redundant numeric signals. Having established scale and dependence, the next step asks which features separate defaulted from non-defaulted observations without using held-out outcomes.

## 6. Inspect categorical support and outcome rates

Operational segments can be modelling candidates and later monitoring slices. Small groups or unstable rates require caution even when aggregate performance is strong.

In [ ]:
product_outcomes = pre_holdout.groupby("PRODUCT_TYPE").agg(
    OBSERVATION_COUNT=("PRODUCT_TYPE", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
    DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index().sort_values("DEFAULT_RATE", ascending=False)
product_outcomes

In [ ]:
channel_outcomes = pre_holdout.groupby("ORIGINATION_CHANNEL").agg(
    OBSERVATION_COUNT=("ORIGINATION_CHANNEL", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
    DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index().sort_values("DEFAULT_RATE", ascending=False)
channel_outcomes

In [ ]:
tenure_outcomes = pre_holdout.groupby("TENURE_BAND").agg(
    OBSERVATION_COUNT=("TENURE_BAND", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
    DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index().sort_values("DEFAULT_RATE", ascending=False)
tenure_outcomes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for data, category, title, axis in (
    (product_outcomes, "PRODUCT_TYPE", "Product", axes[0]),
    (channel_outcomes, "ORIGINATION_CHANNEL", "Origination channel", axes[1]),
    (tenure_outcomes, "TENURE_BAND", "Tenure", axes[2]),
):
    sns.barplot(data=data, x=category, y="DEFAULT_RATE", ax=axis, color="steelblue")
    axis.set_title(f"Default rate by {title.lower()}")
    axis.set_xlabel(title)
    axis.set_ylabel("Finalised default rate")
    axis.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

The target comparisons identify candidate signal. Before carrying categorical fields forward, verify that each segment has enough accounts, observations, and defaults to support estimation. The intervals below are descriptive observation-level intervals only; repeated monthly observations from the same account are correlated, so Phase 3 must use account-aware uncertainty estimates.

In [ ]:
segment_support = pd.concat([
    product_outcomes.rename(columns={"PRODUCT_TYPE": "SEGMENT_VALUE"}).assign(SEGMENT_TYPE="PRODUCT_TYPE"),
    channel_outcomes.rename(columns={"ORIGINATION_CHANNEL": "SEGMENT_VALUE"}).assign(SEGMENT_TYPE="ORIGINATION_CHANNEL"),
    tenure_outcomes.rename(columns={"TENURE_BAND": "SEGMENT_VALUE"}).assign(SEGMENT_TYPE="TENURE_BAND"),
], ignore_index=True)

segment_support["OBSERVATION_LEVEL_STANDARD_ERROR"] = np.sqrt(
    segment_support["DEFAULT_RATE"] * (1 - segment_support["DEFAULT_RATE"])
    / segment_support["OBSERVATION_COUNT"]
)
segment_support["OBSERVATION_LEVEL_CI_LOW"] = segment_support["DEFAULT_RATE"] - (
    1.96 * segment_support["OBSERVATION_LEVEL_STANDARD_ERROR"]
)
segment_support["OBSERVATION_LEVEL_CI_HIGH"] = segment_support["DEFAULT_RATE"] + (
    1.96 * segment_support["OBSERVATION_LEVEL_STANDARD_ERROR"]
)
segment_support.sort_values(["SEGMENT_TYPE", "DEFAULT_RATE"], ascending=[True, False])

## 7. Examine feature stability before the held-out period

A feature can be predictive yet operationally fragile. Compare development with the later validation period while leaving observations from 2026 onward untouched for final evaluation.

In [ ]:
scenario_data = pre_holdout.copy()
scenario_data["ANALYSIS_PERIOD"] = "DEVELOPMENT"
scenario_data.loc[scenario_data["OBSERVATION_DATE"] >= development_cutoff, "ANALYSIS_PERIOD"] = "VALIDATION"

scenario_profile = scenario_data.groupby("ANALYSIS_PERIOD").agg(
    OBSERVATION_COUNT=("ANALYSIS_PERIOD", "count"),
    DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
    AVG_UTILISATION_RATIO=("UTILISATION_RATIO", "mean"),
    AVG_PAYMENT_AMOUNT_30D=("PAYMENT_AMOUNT_30D", "mean"),
    MISSED_PAYMENT_RATE_30D=("MISSED_PAYMENT_30D", "mean"),
    AVG_MISSED_PAYMENTS_6M=("MISSED_PAYMENTS_6M", "mean"),
    ARREARS_RATE_30D=("IN_ARREARS_30D", "mean"),
    AVG_CUSTOMER_CONTACTS_90D=("CUSTOMER_CONTACTS_90D", "mean"),
).reset_index().sort_values("ANALYSIS_PERIOD")
scenario_profile

In [ ]:
scenario_long = scenario_profile.melt(
    id_vars=["ANALYSIS_PERIOD"],
    value_vars=[
        "AVG_UTILISATION_RATIO",
        "MISSED_PAYMENT_RATE_30D",
        "AVG_MISSED_PAYMENTS_6M",
        "ARREARS_RATE_30D",
    ],
    var_name="MEASURE",
    value_name="VALUE",
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=scenario_long, x="MEASURE", y="VALUE", hue="ANALYSIS_PERIOD", ax=ax)
ax.set_title("Feature behaviour across development and validation")
ax.set_xlabel("Measure")
ax.set_ylabel("Mean or rate")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

The mean comparison is only a first stability check. The next outputs examine low, median, and high quantiles, category proportions, and period-specific missingness before drawing a conclusion.

In [ ]:
stability_q05 = scenario_data.groupby("ANALYSIS_PERIOD")[numeric_cols].quantile(0.05).add_suffix("_Q05")
stability_q50 = scenario_data.groupby("ANALYSIS_PERIOD")[numeric_cols].quantile(0.50).add_suffix("_Q50")
stability_q95 = scenario_data.groupby("ANALYSIS_PERIOD")[numeric_cols].quantile(0.95).add_suffix("_Q95")
stability_quantiles = stability_q05.join(stability_q50).join(stability_q95).reset_index()
stability_quantiles

In [ ]:
category_mix = pd.concat([
    scenario_data.groupby(["ANALYSIS_PERIOD", "PRODUCT_TYPE"]).size().rename("OBSERVATION_COUNT").reset_index().assign(SEGMENT_TYPE="PRODUCT_TYPE").rename(columns={"PRODUCT_TYPE": "SEGMENT_VALUE"}),
    scenario_data.groupby(["ANALYSIS_PERIOD", "ORIGINATION_CHANNEL"]).size().rename("OBSERVATION_COUNT").reset_index().assign(SEGMENT_TYPE="ORIGINATION_CHANNEL").rename(columns={"ORIGINATION_CHANNEL": "SEGMENT_VALUE"}),
    scenario_data.groupby(["ANALYSIS_PERIOD", "TENURE_BAND"]).size().rename("OBSERVATION_COUNT").reset_index().assign(SEGMENT_TYPE="TENURE_BAND").rename(columns={"TENURE_BAND": "SEGMENT_VALUE"}),
], ignore_index=True)
period_totals = category_mix.groupby(["ANALYSIS_PERIOD", "SEGMENT_TYPE"]).agg(
    PERIOD_TOTAL=("OBSERVATION_COUNT", "sum")
).reset_index()
category_mix = category_mix.merge(period_totals, on=["ANALYSIS_PERIOD", "SEGMENT_TYPE"], how="left")
category_mix["SEGMENT_SHARE"] = category_mix["OBSERVATION_COUNT"] / category_mix["PERIOD_TOTAL"]
category_mix.sort_values(["SEGMENT_TYPE", "SEGMENT_VALUE", "ANALYSIS_PERIOD"])

In [ ]:
period_counts = scenario_data.groupby("ANALYSIS_PERIOD")[numeric_cols].count()
period_sizes = scenario_data.groupby("ANALYSIS_PERIOD").size().rename("PERIOD_ROW_COUNT").reset_index()
period_missingness = period_counts.reset_index().merge(period_sizes, on="ANALYSIS_PERIOD", how="left")
for column in numeric_cols:
    period_missingness[f"{column}_NULL_COUNT"] = period_missingness["PERIOD_ROW_COUNT"] - period_missingness[column]
period_missingness = period_missingness[[
    "ANALYSIS_PERIOD", "PERIOD_ROW_COUNT",
    *[f"{column}_NULL_COUNT" for column in numeric_cols],
]]
period_missingness

Review the means, quantiles, category shares, and missingness together. If they agree that development and validation are similar, the first feature contract is reasonably stable before synthetic drift begins. Material movement in any one view should instead trigger feature-specific investigation before Phase 3.

## 8. Define temporal development, validation, and held-out windows

Use contiguous observation periods. Development ends before validation, and the held-out test begins at the controlled drift boundary. No model or feature decision should use held-out outcomes.

In [ ]:
splits = traning_base_pd.copy()
splits["SPLIT_NAME"] = "PURGED_OR_HELD_OUT"
splits.loc[eligible_development, "SPLIT_NAME"] = "DEVELOPMENT"
splits.loc[eligible_validation, "SPLIT_NAME"] = "VALIDATION"
splits.loc[splits["OBSERVATION_DATE"] >= validation_cutoff, "SPLIT_NAME"] = "HELD_OUT_TEST"

split_stats = splits.groupby("SPLIT_NAME").agg(
    FIRST_OBSERVATION=("OBSERVATION_DATE", "min"),
    LAST_OBSERVATION=("OBSERVATION_DATE", "max"),
    OBSERVATION_COUNT=("SPLIT_NAME", "count"),
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
).reset_index()

visible = splits[splits["SPLIT_NAME"].isin(["DEVELOPMENT", "VALIDATION"])].groupby("SPLIT_NAME").agg(
    VISIBLE_DEFAULT_COUNT=("DEFAULT_WITHIN_90D", "sum"),
    VISIBLE_DEFAULT_RATE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index()

split_assignment = split_stats.merge(visible, on="SPLIT_NAME", how="left")
split_assignment.sort_values("FIRST_OBSERVATION")

The split table makes the 90-day embargo visible as `PURGED_OR_HELD_OUT`: rows whose outcomes cross a decision boundary are not used for fitting or validation. Repeated accounts across eligible periods reflect the operational task of rescoring existing accounts, while the held-out target remains hidden. Phase 3 should also report a new-account sensitivity analysis to separate temporal generalisation from borrower memorisation.

## 9. Translate prevalence into monthly review capacity

Accuracy is not an informative primary metric for an imbalanced target. The operational baseline is historical prevalence, but review capacity applies to a monthly account queue rather than all account-month rows pooled across a split. Calculate the 10% review budget for each eligible scoring month, then summarise that operational demand by split.

In [ ]:
eligible_splits = splits[splits["SPLIT_NAME"].isin(["DEVELOPMENT", "VALIDATION"])]
monthly_review_capacity = eligible_splits.groupby(["SPLIT_NAME", "OBSERVATION_DATE"]).agg(
    ACCOUNT_COUNT=("ACCOUNT_ID", "nunique"),
    PREVALENCE_BASELINE=("DEFAULT_WITHIN_90D", "mean"),
).reset_index()
monthly_review_capacity["TARGET_REVIEW_RATE"] = 0.10
monthly_review_capacity["TARGET_REVIEW_COUNT"] = np.ceil(
    monthly_review_capacity["ACCOUNT_COUNT"] * monthly_review_capacity["TARGET_REVIEW_RATE"]
).astype(int)
monthly_review_capacity.sort_values(["SPLIT_NAME", "OBSERVATION_DATE"])

In [ ]:
review_capacity_summary = monthly_review_capacity.groupby("SPLIT_NAME").agg(
    SCORING_MONTH_COUNT=("OBSERVATION_DATE", "nunique"),
    AVG_MONTHLY_ACCOUNT_COUNT=("ACCOUNT_COUNT", "mean"),
    AVG_MONTHLY_REVIEW_COUNT=("TARGET_REVIEW_COUNT", "mean"),
    MIN_MONTHLY_PREVALENCE=("PREVALENCE_BASELINE", "min"),
    AVG_MONTHLY_PREVALENCE=("PREVALENCE_BASELINE", "mean"),
    MAX_MONTHLY_PREVALENCE=("PREVALENCE_BASELINE", "max"),
).reset_index()
review_capacity_summary

Phase 3 will compare candidates using:

- ROC AUC for ranking across thresholds.
- Average precision for minority-class ranking quality.
- Brier score and calibration evidence for probability quality.
- Recall among the highest-risk 10% of accounts in each scoring month, summarised across months.
- Segment ROC AUC and support for product, origination channel, and tenure.
- A new-account sensitivity result alongside the primary forward-time existing-account evaluation.

Configured candidate gates are ROC AUC ≥ 0.72, average precision ≥ 0.20, Brier score ≤ 0.16, and segment ROC AUC ≥ 0.62. These are demonstration thresholds and must not be treated as real credit-risk policy.

## 10. Provisional feature decision

**Include for the first candidate:**

- Categorical: `PRODUCT_TYPE`, `ORIGINATION_CHANNEL`, `TENURE_BAND`.
- Numeric: `ACCOUNT_AGE_MONTHS`, `CREDIT_LIMIT`, `UTILISATION_RATIO`, `MONTHLY_INCOME_ESTIMATE`, `PAYMENT_AMOUNT_30D`, `MISSED_PAYMENT_30D`, `MISSED_PAYMENTS_6M`, `IN_ARREARS_30D`, `CUSTOMER_CONTACTS_90D`.

**Exclude from model inputs:**

- `ACCOUNT_ID`: identifier and memorisation risk.
- `OBSERVATION_DATE`: split/control field; avoid learning a synthetic calendar shortcut in the first candidate.
- `OUTCOME_FINALITY_DATE`: label-availability information and direct leakage risk.
- `DEFAULT_WITHIN_90D`: target.

**Questions carried into Phase 3:**

- Do linear and tree-based candidates react differently to the scale and bounded counts?
- Does origination channel add stable signal or only segment variation?
- How much performance changes between validation and controlled-drift held-out data?
- Are probabilities sufficiently calibrated for prioritisation, or is post-fit calibration required?

This phase ends with a reviewable hypothesis. It does not train or register a model.